In [5]:
# set wd
import os
from pathlib import Path
print(os.getcwd())
wd = Path(os.getcwd())
if wd.name == "notebooks":
    %cd ..
print(f"Working Dir Base: {(os.getcwd())}")

/Users/peli/Projects/Repositories/MEGPypes
Working Dir Base: /Users/peli/Projects/Repositories/MEGPypes


In [6]:
import yaml
import time
from bids.layout import BIDSLayout
# import
from src.pipelines.init_preproc import create_initial_preprocessing

loaded a MEG dataset from here:
https://openneuro.org/datasets/ds006629/versions/1.0.1

using datalad
```
datalad install -s https://github.com/OpenNeuroDatasets/ds006629.git data/ds006629
```
```
cd output/ds006629
datalad get -r .
```


First we need to grab data from our dataset.
We assume the data operate within a bids compliant data structure
https://peerherholz.github.io/workshop_weizmann/nipype/notebooks/basic_data_input_bids.html



In [7]:
# do some data inspection using pybids?
layout = BIDSLayout("data/ds006629/")
print(layout)
# subjects
subjects = layout.get_subjects()
print(f"Subjects: {subjects}")
# datatypes
bidstypes = layout.get_datatypes()
print(f"Data types: {bidstypes}")
# suffixes
print(layout.get_suffixes(datatype='func'))
# see tasks
layout.get_tasks()
# see dataset description
layout.get_dataset_description()
#

# see data metadata


BIDS Layout: ...itories/MEGPypes/data/ds006629 | Subjects: 19 | Sessions: 0 | Runs: 19
Subjects: ['01', '02', '04', '05', '06', '07', '08', '09', '10', '11', '12', '14', '15', '16', '17', '18', '19', '20', '21']
Data types: ['meg']
[]


{'Name': 'SINGSING',
 'BIDSVersion': '1.7.0',
 'License': 'CC0',
 'DatasetType': 'raw',
 'Authors': ['Valerie Chanoine',
  'Jean-Michel Badier',
  'Mireille Besson',
  'Talya Inbar'],
 'Acknowledgements': 'MEG data acquisition was performed in the MEG Centre (Timone Hospital, Marseille, France)',
 'Funding': ['This research has been supported by funding from the Institute of Convergence ILCB (France 2030, ANR-16-CONV-0002) and the Excellence Initiative of Aix-Marseille University A*MIDEX (ANR-11-IDEX-0001-02)'],
 'ReferencesAndLinks': ['a data paper',
  'a resource to be cited when using the data'],
 'DatasetDOI': 'doi:10.18112/openneuro.ds006629.v1.0.1',
 'GeneratedBy': [{'Name': 'MNE-BIDS',
   'Version': '0.14',
   'Description': 'MNE-BIDS is a Python package that allows you to read and write BIDS-compatible datasets with the help of MNE-Python.'}],
 'SourceDatasets': [{'DOI': 'doi:10.18112/openneuro.ds006629.v1.0.0',
   'URL': 'https://openneuro.org/datasets/ds006629',
   'Version':

In [8]:
# Load configs
import os
import yaml
from nipype import config as nconfig

config_path = "config/config_ds006629.yaml"
with open(config_path, "r") as yamlfile:
    config = yaml.load(yamlfile, Loader=yaml.FullLoader)

paths_config = config["paths"]
proc_config = config["processing"]
steps_config = config["steps"]
wf_config = config["workflow"]

# Configure Nipype logging (applies to all subprocesses)
nconfig.update_config({
    'logging': {
        'log_directory': os.path.join(paths_config["workdir"], 'logs'),
        'log_to_file': True,
        'interface_level': 'info',
        'workflow_level': 'info',
    },
    'execution': {
        'crashdump_dir': os.path.abspath('crashes'),
        'remove_unnecessary_outputs': False,
    }
})

# Create workflow
wf = create_initial_preprocessing(
    basedir=paths_config["basedir"],
    workdir=paths_config["workdir"],
    output_dir=paths_config["outputdir"],
    subject_list=paths_config["subjects"],  # ← Now actually used!
    stepflags_params=steps_config,
    crop_params=proc_config["crop"],
    filter_params=proc_config["filter"],
    gradcomp_params=proc_config["gradcomp"],
    ica_params=proc_config["ica"]
)

# visualize workflow graph
wf.write_graph(graph2use='colored', simple_form=True)
print(f"Workflow graph saved to: {wf.base_dir}/megpreproc/graph.png")

# Run workflow
n_workers = wf_config.get("n_workers", max(1, os.cpu_count() - 2))
print(f"Running with {n_workers} workers")

result = wf.run(
    plugin=wf_config["plugin"],
    plugin_args={"n_procs": n_workers}
)

raw_dir /Users/peli/Projects/Repositories/MEGPypes/data/ds006629
260227-12:42:42,275 nipype.workflow INFO:
	 Generated workflow graph: workdir/megpreproc/graph.png (graph2use=colored, simple_form=True).
Workflow graph saved to: workdir/megpreproc/graph.png
Running with 8 workers
260227-12:42:42,277 nipype.workflow INFO:
	 Workflow megpreproc settings: ['check', 'execution', 'logging', 'monitoring']
260227-12:42:42,281 nipype.workflow INFO:
	 Running serially.
260227-12:42:42,281 nipype.workflow INFO:
	 [Node] Setting-up "megpreproc.selectfiles" in "/Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-01/selectfiles".
260227-12:42:42,283 nipype.workflow INFO:
	 [Node] Executing "selectfiles" <nipype.interfaces.io.SelectFiles>
260227-12:42:42,283 nipype.workflow INFO:
	 [Node] Finished "selectfiles", elapsed time 0.000132s.
260227-12:42:42,285 nipype.workflow INFO:
	 [Node] Setting-up "megpreproc.selectfiles" in "/Users/peli/Projects/Repositories/MEGPypes/workdir